In [61]:
''' 
This document demonstrates how to use LangChain with a PDF document.

'''

from langchain_core.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore



'''Docling has a better answer so far'''
from langchain_docling import DoclingLoader

import getpass
import os


if not os.environ.get("GROQ_API_KEY"):
  os.environ["GROQ_API_KEY"] = getpass.getpass("Enter API key for GROQ_API_KEY: ")

from langchain.chat_models import init_chat_model

llm = init_chat_model("meta-llama/llama-4-scout-17b-16e-instruct", model_provider="groq")
# llm = init_chat_model("gpt-4.1-nano", model_provider="openai", temperature=0)
# llm = init_chat_model("llama3.1", model_provider="ollama")
# llm = init_chat_model("deepseek-r1", model_provider="ollama")
llm.temperature = 0

In [3]:
def get_pdf_content(file_path):
    loader = DoclingLoader(file_path)
    docs = []
    for doc in loader.lazy_load():
        docs.append(doc)
    return "".join(doc.page_content for doc in docs), docs

In [4]:
import chromadb

persistent_client = chromadb.PersistentClient()
try:
    collection = persistent_client.get_collection(name="MOSFETs")
    print(f"Collection found.")
except:
    collection = None
    print(f"Collection not found. Please create one.")


Collection found.


# we laod the document content here. 
if it's already loaded it won't be loaded again

In [5]:
document = 'tms320f28069m-q1'
# document = 'LAUNCHXL-F28069M'
# document = 'TI_csd19538q2'
# document = 'Infineon_IMZC120R017M2H'
# document = 'Wolfspeed_C3M0016120K'
# document = 'BOOSTXL-DRV8305EVM'

if not collection:
    # Create a new collection
    collection = persistent_client.create_collection(name="MOSFETs")
    print(f"Collection {document} created.")
# Add documents to the collection
doc_data = collection.get(ids=[document])
if not doc_data['ids']:
    file_path = f"./docs/{document}.pdf"
    context, docs = get_pdf_content(file_path)
    collection.add(
        documents=[context],
        metadatas=[{"source": file_path}],
        ids=[document],
    )
docs_content = doc_data["documents"]

limit   = 500    # tune this to balance request‑size vs. memory
offset  = 0
results = collection.get(
    limit=limit,
    offset=offset,
)
print(f"Found {len(results['documents'])} documents.")

Found 6 documents.


# use RAG if the document is large

In [42]:
use_RAG = True
if use_RAG:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=128000, chunk_overlap=2000, add_start_index=True
    )
    # _, docs = get_pdf_content(file_path)
    all_splits = text_splitter.split_text(docs_content[0])

    print(len(all_splits))

    '''Embeddings'''
    embeddings = OpenAIEmbeddings(model="text-embedding-3-large")


    '''Storing'''
    vector_store = InMemoryVectorStore(embeddings)
    # Add documents with generated ids
    ids = vector_store.add_texts(texts=all_splits)
else:
    pass

6


In [62]:

# llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")
# llm = init_chat_model("o3-mini", model_provider="openai")
# llm = init_chat_model("o4-mini", model_provider="openai")
llm = init_chat_model("claude-3-7-sonnet-latest", model_provider="anthropic")
# llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)
# llm = init_chat_model("llama3.1", model_provider="ollama")
# llm = init_chat_model("deepseek-r1", model_provider="ollama")
# llm.temperature = 0

In [63]:
question = "How can I measure a voltage differentially using ADCs in this micro?"
if use_RAG:
    embedded_query = embeddings.embed_query(question)
    results = vector_store.similarity_search_with_score_by_vector(embedded_query, k=1)
    context = [doc.page_content for doc, score in results]
    # retrieved_docs = vector_store.similarity_search(question)
    # retrieved_docs = vector_store.similarity_search_with_score(question, k=1)
    # context = [doc.page_content for doc in retrieved_docs]
else:
    context = docs_content
template = """Use the following pieces of context to answer the question at the end.
{context}
Question: {question}
If the question is not answerable based on the context, please say "I don't know".
"""
custom_rag_prompt = PromptTemplate.from_template(template)

prompt = custom_rag_prompt.invoke({"context": context, "question": question})
answer = llm.invoke(prompt)
print(f"response: {answer.content}") 


response: Based on the provided context, there is no explicit information about using ADCs in this microcontroller to measure voltages differentially. While the document contains detailed information about the ADC modules, including specifications, timing requirements, electrical characteristics, and configuration, it does not specifically describe differential voltage measurement techniques or configurations for the ADCs.

The ADC section describes single-ended inputs, references, and various other features, but does not mention differential measurement capability. The document shows that the ADC has 16 multiplexed input channels and sample-and-hold circuits, but doesn't indicate they can be configured for differential measurements.

I don't know how to measure a voltage differentially using the ADCs in this microcontroller based on the information provided.


## To get answer in structured format, use the following code

In [ ]:

from pydantic import BaseModel, Field
class ResponseFormatter(BaseModel):
    """Always use this tool to structure your response to the user."""
    min: str = Field(description="value for the minimum")
    max: str = Field(description="value for the maximum")
    typical: str = Field(description="value for the typical")
    unit: str = Field(description="Unit of the value")
    temperature: str = Field(description="Temperature of the value")
llm=llm.bind_tools([ResponseFormatter])

ai_msg = llm.invoke(f"get the threshold values from {answer.content} and return them in the format of the ResponseFormatter tool")

pydantic_object = ResponseFormatter.model_validate(ai_msg.tool_calls[0]["args"])

print(f"{document} vth sepcs are:\n {pydantic_object}")    